# Recursive CTEs

## Overview
A **recursive CTE** is a CTE that references itself. It enables SQL to traverse hierarchies, walk graphs, and generate sequences — things impossible with regular SQL.

**Structure:**
```sql
WITH RECURSIVE cte_name AS (
    -- Anchor member: the starting point (non-recursive)
    SELECT base_case_rows

    UNION ALL

    -- Recursive member: references cte_name, adds one more level
    SELECT next_level_rows
    FROM   source_table
    JOIN   cte_name ON join_condition
    WHERE  termination_condition    -- must eventually be false to stop
)
SELECT * FROM cte_name;
```

**Execution model:**
1. Evaluate the anchor member → initial result set (Level 0)
2. Evaluate the recursive member using Level 0 as input → Level 1
3. Evaluate the recursive member using Level 1 as input → Level 2
4. Repeat until the recursive member returns zero rows
5. UNION ALL all levels together

**`UNION ALL` vs `UNION`:** Recursive CTEs use `UNION ALL` (keeps duplicates, faster). `UNION` is allowed but deduplicates at each step — very slow and rarely needed.

**Termination:** The recursive member must eventually produce zero rows. A missing or wrong WHERE condition causes an infinite loop. Most databases have a maximum recursion depth (PostgreSQL default: 100; configurable).

**SQLite:** Supported since 3.8.3. Requires `WITH RECURSIVE` — unlike PostgreSQL where `WITH` alone is sufficient.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, active INTEGER DEFAULT 1);
CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT, province TEXT, manager_id INTEGER);
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_id INTEGER);
CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER, dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER);
CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT, category TEXT);
CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER, test_name TEXT, result_val REAL, units TEXT, collected TEXT, flagged INTEGER);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),
  (5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),
  (7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),
  (9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),
  (11,'Diana','Flores','2000-02-14','F','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),(11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Osei','Family Medicine','ON',10),(13,'Dr. Ethan Larson','Orthopaedics','QC',10),
  (14,'Dr. Priya Sharma','Emergency','AB',10),(15,'Dr. Lucas Petit','Radiology','QC',11);
INSERT INTO departments VALUES
  (1,'Emergency','Tower A',14),(2,'Cardiology','Tower B',10),(3,'Oncology','Tower C',11),
  (4,'Family Medicine','Clinic',12),(5,'Orthopaedics','Tower A',13),
  (6,'Radiology','Tower B',15),(7,'ICU','Tower C',NULL);
INSERT INTO encounters VALUES
  (1,1,10,2,'2023-01-15','I25.1',3200.00,1),(2,2,14,1,'2023-02-20','J18.9',1850.00,1),
  (3,3,12,4,'2023-03-05','Z00.0',120.00,0),(4,4,13,5,'2023-03-18','M17.1',5500.00,1),
  (5,5,12,4,'2023-04-02','J06.9',95.00,0),(6,6,14,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,2,'2023-06-22','I10',2100.00,1),(8,8,12,4,'2023-07-14',NULL,80.00,0),
  (9,1,14,1,'2023-08-30','R55',410.00,0),(10,3,12,4,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,2,'2023-10-01','I48.0',1750.00,1),(12,10,14,1,'2023-11-03','S52.5',2200.00,1),
  (13,2,10,2,'2023-11-20','I25.1',2900.00,1),(14,6,12,4,'2023-12-01','Z00.0',115.00,0);
INSERT INTO diagnoses VALUES
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),
  ('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),
  ('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),
  ('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');
INSERT INTO lab_results VALUES
  (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),
  (3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),
  (5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),
  (7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),
  (9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),
  (11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);
""")
conn.commit()
print('Healthcare schema ready.')
print()
print('Provider hierarchy (manager_id is self-referencing):')
print(pd.read_sql('SELECT provider_id, full_name, specialty, manager_id FROM providers', conn).to_string(index=False))


Healthcare schema ready.

Provider hierarchy (manager_id is self-referencing):
 provider_id         full_name       specialty  manager_id
          10 Dr. Sarah MacNeil      Cardiology         NaN
          11    Dr. James Wong        Oncology        10.0
          12   Dr. Fatima Osei Family Medicine        10.0
          13  Dr. Ethan Larson    Orthopaedics        10.0
          14  Dr. Priya Sharma       Emergency        10.0
          15   Dr. Lucas Petit       Radiology        11.0


---
## Traversing an org chart — top-down

In [2]:
# Start from the top-level provider (manager_id IS NULL) and walk down
print('=== Provider org chart — top-down traversal ===')
q = """
WITH RECURSIVE org_chart AS (
    -- Anchor: top-level provider (no manager)
    SELECT  provider_id,
            full_name,
            specialty,
            manager_id,
            0                    AS level,
            full_name            AS path
    FROM    providers
    WHERE   manager_id IS NULL

    UNION ALL

    -- Recursive: each direct report of the current level
    SELECT  p.provider_id,
            p.full_name,
            p.specialty,
            p.manager_id,
            oc.level + 1,
            oc.path || ' > ' || p.full_name
    FROM    providers   AS p
    JOIN    org_chart   AS oc ON p.manager_id = oc.provider_id
)
SELECT  level,
        REPEAT('  ', level) || full_name   AS indented_name,
        specialty,
        path
FROM    org_chart
ORDER BY path
"""
# SQLite doesn't have REPEAT() — use SUBSTR workaround
q_sqlite = q.replace("REPEAT('  ', level)", "SUBSTR('          ', 1, level*2)")
print(pd.read_sql(q_sqlite, conn).to_string(index=False))


=== Provider org chart — top-down traversal ===
 level       indented_name       specialty                                                 path
     0   Dr. Sarah MacNeil      Cardiology                                    Dr. Sarah MacNeil
     1    Dr. Ethan Larson    Orthopaedics                 Dr. Sarah MacNeil > Dr. Ethan Larson
     1     Dr. Fatima Osei Family Medicine                  Dr. Sarah MacNeil > Dr. Fatima Osei
     1      Dr. James Wong        Oncology                   Dr. Sarah MacNeil > Dr. James Wong
     2     Dr. Lucas Petit       Radiology Dr. Sarah MacNeil > Dr. James Wong > Dr. Lucas Petit
     1    Dr. Priya Sharma       Emergency                 Dr. Sarah MacNeil > Dr. Priya Sharma


---
## Bottom-up traversal — finding ancestors

In [3]:
# Start from a specific provider and walk UP to the root
print('=== Find all ancestors of Dr. Lucas Petit (provider_id 15) ===')
q = """
WITH RECURSIVE ancestors AS (
    -- Anchor: start from the target provider
    SELECT  provider_id,
            full_name,
            manager_id,
            0  AS level
    FROM    providers
    WHERE   provider_id = 15

    UNION ALL

    -- Recursive: go up to the manager
    SELECT  p.provider_id,
            p.full_name,
            p.manager_id,
            a.level + 1
    FROM    providers  AS p
    JOIN    ancestors  AS a ON p.provider_id = a.manager_id
)
SELECT  level,
        provider_id,
        full_name,
        CASE level WHEN 0 THEN '(start)' ELSE 'reports to' END AS relationship
FROM    ancestors
ORDER BY level
"""
print(pd.read_sql(q, conn).to_string(index=False))


=== Find all ancestors of Dr. Lucas Petit (provider_id 15) ===
 level  provider_id         full_name relationship
     0           15   Dr. Lucas Petit      (start)
     1           11    Dr. James Wong   reports to
     2           10 Dr. Sarah MacNeil   reports to


---
## Generating a sequence — integer or date series

In [4]:
# Generate a sequence of integers 1–10
print('=== Generate integers 1 to 10 ===')
q = """
WITH RECURSIVE seq(n) AS (
    SELECT 1                     -- anchor
    UNION ALL
    SELECT n + 1 FROM seq        -- recursive step
    WHERE  n < 10                -- termination condition
)
SELECT n FROM seq
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Generate a date series — all months in 2023
print()
print('=== Generate all months in 2023 ===')
q2 = """
WITH RECURSIVE months(month_start) AS (
    SELECT '2023-01-01'
    UNION ALL
    SELECT DATE(month_start, '+1 month')
    FROM   months
    WHERE  month_start < '2023-12-01'
)
SELECT  month_start,
        STRFTIME('%Y-%m', month_start) AS year_month
FROM    months
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Generate integers 1 to 10 ===
 n
 1
 2
 3
 4
 5
 6
 7
 8
 9
10

=== Generate all months in 2023 ===
month_start year_month
 2023-01-01    2023-01
 2023-02-01    2023-02
 2023-03-01    2023-03
 2023-04-01    2023-04
 2023-05-01    2023-05
 2023-06-01    2023-06
 2023-07-01    2023-07
 2023-08-01    2023-08
 2023-09-01    2023-09
 2023-10-01    2023-10
 2023-11-01    2023-11
 2023-12-01    2023-12


---
## Practical: encounter count per month using generated date spine

In [5]:
# Use recursive CTE date spine + LEFT JOIN to get gap-free monthly counts
print('=== Monthly encounter counts — zero-filled with recursive date spine ===')
q = """
WITH RECURSIVE months(month_start) AS (
    SELECT '2023-01-01'
    UNION ALL
    SELECT DATE(month_start, '+1 month')
    FROM   months
    WHERE  month_start < '2023-12-01'
)
SELECT  STRFTIME('%Y-%m', m.month_start)       AS month,
        COUNT(e.enc_id)                         AS encounters,
        COALESCE(ROUND(SUM(e.cost_cad),2), 0)  AS total_cost,
        COALESCE(SUM(e.admitted), 0)            AS admissions
FROM    months AS m
LEFT JOIN encounters AS e
    ON  STRFTIME('%Y-%m', e.enc_date) = STRFTIME('%Y-%m', m.month_start)
GROUP BY m.month_start
ORDER BY m.month_start
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Every month appears even when there are zero encounters.')
print('Without the date spine, months with no activity would be missing.')

conn.close()


=== Monthly encounter counts — zero-filled with recursive date spine ===
  month  encounters  total_cost  admissions
2023-01           1      3200.0           1
2023-02           1      1850.0           1
2023-03           2      5620.0           1
2023-04           1        95.0           0
2023-05           1       780.0           0
2023-06           1      2100.0           1
2023-07           1        80.0           0
2023-08           1       410.0           0
2023-09           1       110.0           0
2023-10           1      1750.0           1
2023-11           2      5100.0           2
2023-12           1       115.0           0

Every month appears even when there are zero encounters.
Without the date spine, months with no activity would be missing.


---
## Common Pitfalls

**1. Missing termination condition causes an infinite loop**
The recursive member must include a `WHERE` clause that eventually evaluates to FALSE. `WHERE n < 10` terminates at 10. Without it, the CTE runs until the database hits its maximum recursion depth limit (typically 100 in PostgreSQL) and raises an error.

**2. Using UNION instead of UNION ALL in the recursive member**
`UNION` deduplicates rows at each iteration, which requires sorting the entire result set every step — extremely slow for large hierarchies. Use `UNION ALL` unless you specifically need cycle detection (and even then, use a path array or visited-set approach instead).

**3. Forgetting `RECURSIVE` keyword in SQLite**
SQLite requires `WITH RECURSIVE cte AS (...)`. In PostgreSQL, `WITH cte AS (...)` works for recursive CTEs too (the RECURSIVE keyword is optional there). In SQLite, omitting `RECURSIVE` causes a syntax error or treats the CTE as non-recursive.

**4. Cycles in the graph cause infinite recursion**
If the data has cycles (A → B → C → A), the recursive CTE loops forever until hitting the depth limit. Detect cycles by carrying a path string and checking `INSTR(path, provider_id) = 0`, or use PostgreSQL's array approach: `WHERE provider_id != ALL(visited_array)`.

**5. The anchor and recursive member must have the same column structure**
`SELECT id, name FROM ... UNION ALL SELECT id FROM ...` fails — column counts must match, and types must be compatible. Use `NULL AS col_name` padding where needed.

**6. Recursion depth limits can silently truncate deep hierarchies**
PostgreSQL's default max recursion depth is 100 (`max_recursion_depth`). A 200-level org chart silently stops at 100 levels. Adjust the setting or restructure the data. For very deep hierarchies (graph databases), Neo4j (covered in `16_graph_databases_neo4j`) handles arbitrary depth more naturally.


---
*sql_methods_library - Samantha McGarrigle*